In [1]:
!pip install -q git+https://github.com/abhibarnwal707/PaperTrail.git
import os
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
import dagshub, mlflow
dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 83.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/2

Accessing as abhibarnwal707

Initialized MLflow to track repo "abhibarnwal707/papertrail"

Repository abhibarnwal707/papertrail initialized!

In [3]:
# load, split, build all 51 features once
import glob
import json
import time
import numpy as np
import polars as pl
import xgboost as xgb

from papertrail.features import add_out_count_24h, build_features, fit_categories
from papertrail.graph_features import HISTORY_FEATURES, add_history_features
from papertrail.metrics import best_f1_threshold, evaluate
from papertrail.split import assert_valid_split, split_summary, temporal_split

path = glob.glob("/kaggle/input/datasets/abhibarnwal707/papertrail-processed/transactions.parquet", recursive=True)[0]
df = pl.read_parquet(path).sort(["ts", "txn_id"])
df = add_out_count_24h(df)
summary = split_summary(df)
print(summary)
assert_valid_split(summary)

t0 = time.time()
dfx = add_history_features(df)
print(f"history features in {time.time() - t0:.0f}s ->", dfx.shape)

train, val, test = temporal_split(dfx)
cats = fit_categories(train)


def to_matrix(part):
    base = build_features(part, cats)
    return pl.concat([base, part.select(HISTORY_FEATURES)], how="horizontal").cast(pl.Float32)


Xtr_df, Xva_df, Xte_df = to_matrix(train), to_matrix(val), to_matrix(test)
ALL = Xtr_df.columns
TABULAR = [c for c in ALL if c not in HISTORY_FEATURES]
X_train, X_val, X_test = Xtr_df.to_numpy(), Xva_df.to_numpy(), Xte_df.to_numpy()
y_train, y_val, y_test = train["label"].to_numpy(), val["label"].to_numpy(), test["label"].to_numpy()
spw = float((y_train == 0).sum() / y_train.sum())
ach_test = (test["pay_format"] == "ACH").to_numpy()
print(len(ALL), "features,", len(TABULAR), "tabular; scale_pos_weight =", round(spw, 1))


shape: (4, 6)
┌──────────────┬─────────┬───────────┬──────────┬─────────────────────┬─────────────────────┐
│ split        ┆ rows    ┆ positives ┆ pos_rate ┆ start               ┆ end                 │
│ ---          ┆ ---     ┆ ---       ┆ ---      ┆ ---                 ┆ ---                 │
│ str          ┆ i64     ┆ i64       ┆ f64      ┆ datetime[μs]        ┆ datetime[μs]        │
╞══════════════╪═════════╪═══════════╪══════════╪═════════════════════╪═════════════════════╡
│ train        ┆ 3248921 ┆ 2530      ┆ 0.000779 ┆ 2022-09-01 00:00:00 ┆ 2022-09-06 23:59:00 │
│ val          ┆ 965524  ┆ 1036      ┆ 0.001073 ┆ 2022-09-07 00:00:00 ┆ 2022-09-08 23:59:00 │
│ test         ┆ 862792  ┆ 956       ┆ 0.001108 ┆ 2022-09-09 00:00:00 ┆ 2022-09-10 23:59:00 │
│ dropped_tail ┆ 1108    ┆ 655       ┆ 0.591155 ┆ 2022-09-11 00:05:00 ┆ 2022-09-18 16:18:00 │
└──────────────┴─────────┴───────────┴──────────┴─────────────────────┴─────────────────────┘
history features in 45s -> (5078345, 52)
51 fe

In [4]:
# feature families (a feature belongs to exactly one family)
def fam_of(col):
    base = col[2:] if col[:2] in ("s_", "d_") else col
    if base in ("uniq_out", "uniq_in"):
        return "uniq"                         # distinct counterparties so far
    if base in ("tot_out", "tot_in", "gap_s", "age_s"):
        return "totals_timing"                # all-time counts, time since last event, account age
    if base == "has_reverse_before" or base.endswith("pagerank"):
        return "graph"                        # reciprocal edge + daily PageRank
    if base.startswith(("out_", "in_")):
        return "windowed"                     # 24h / 7d counts, sums, means
    raise ValueError(col)


FAMILIES = {}
for c in HISTORY_FEATURES:
    FAMILIES.setdefault(fam_of(c), []).append(c)
assert sum(len(v) for v in FAMILIES.values()) == len(HISTORY_FEATURES) == 35
for k, v in FAMILIES.items():
    print(f"{k:14s} {len(v):2d}  {v if len(v) <= 6 else v[:3] + ['...']}")
SIDE = {"s_": [c for c in HISTORY_FEATURES if c.startswith("s_")],
        "d_": [c for c in HISTORY_FEATURES if c.startswith("d_")]}

# arm name -> (feature list, seed)
ARMS = {"tabular_only": (TABULAR, 0), "full": (ALL, 0), "full_seed1": (ALL, 1), "full_seed2": (ALL, 2)}
for fam, cols in FAMILIES.items():
    ARMS[f"tab+{fam}"] = (TABULAR + cols, 0)                          # add one family
    ARMS[f"full-{fam}"] = ([c for c in ALL if c not in cols], 0)       # remove one family
ARMS["full-sender_side"] = ([c for c in ALL if c not in SIDE["s_"]], 0)
ARMS["full-receiver_side"] = ([c for c in ALL if c not in SIDE["d_"]], 0)
ARMS["tab+uniq_out_only"] = (TABULAR + ["s_uniq_out", "d_uniq_out"], 0)   # the single top feature, both sides
print(len(ARMS), "arms:", list(ARMS))


windowed       20  ['s_out_cnt_24h', 's_out_cnt_7d', 's_in_cnt_24h', '...']
uniq            4  ['s_uniq_out', 's_uniq_in', 'd_uniq_out', 'd_uniq_in']
totals_timing   8  ['s_tot_out', 's_tot_in', 's_gap_s', '...']
graph           3  ['has_reverse_before', 's_pagerank', 'd_pagerank']
15 arms: ['tabular_only', 'full', 'full_seed1', 'full_seed2', 'tab+windowed', 'full-windowed', 'tab+uniq', 'full-uniq', 'tab+totals_timing', 'full-totals_timing', 'tab+graph', 'full-graph', 'full-sender_side', 'full-receiver_side', 'tab+uniq_out_only']


In [6]:
# %% Cell 4: evaluation helper + one-arm fit (same protocol and params as 03_graph_feats)
def run_eval(p_val, p_test):
    thr = best_f1_threshold(y_val, p_val)
    res = {"threshold": float(thr)}
    for k, v in evaluate(y_val, p_val, thr).items():
        res[f"val_{k}"] = v
    for k, v in evaluate(y_test, p_test, thr).items():
        res[f"test_{k}"] = v
    for k, v in evaluate(y_test[ach_test], p_test[ach_test], thr).items():
        res[f"test_ach_{k}"] = v
    return res


def fit_arm(name, cols, seed):
    idx = [ALL.index(c) for c in cols]
    params = dict(tree_method="hist", n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8,
                  colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr", early_stopping_rounds=50,
                  n_jobs=-1, random_state=seed)
    t0 = time.time()
    model = xgb.XGBClassifier(**params)
    model.fit(X_train[:, idx], y_train, eval_set=[(X_val[:, idx], y_val)], verbose=False)
    p_val = model.predict_proba(X_val[:, idx])[:, 1]
    p_test = model.predict_proba(X_test[:, idx])[:, 1]
    res = run_eval(p_val, p_test)
    res.update(n_features=len(cols), best_iteration=int(model.best_iteration), seconds=round(time.time() - t0))
    with mlflow.start_run(run_name=f"abl_{name}"):
        mlflow.log_params({"arm": name, "seed": seed, "n_features": len(cols)})
        mlflow.log_metrics({k: v for k, v in res.items() if k != "n_features"})
    return res, p_test


In [7]:
# %% Cell 5: run the arms (about 16 fits; set ONLY to a list of arm names to run a subset)
ONLY = None                        # e.g. ["tabular_only", "full", "full-uniq"]
results, test_preds = {}, {}
for name, (cols, seed) in ARMS.items():
    if ONLY and name not in ONLY:
        continue
    results[name], test_preds[name] = fit_arm(name, cols, seed)
    r = results[name]
    print(f"{name:22s} nfeat={r['n_features']:2d}  val_prauc={r['val_pr_auc']:.3f}  test_prauc={r['test_pr_auc']:.3f}  "
          f"f1={r['test_f1']:.3f}  rec@1%={r['test_recall_at_1pct_fpr']:.3f}  p@1000={r['test_precision_at_1000']:.3f}  "
          f"iter={r['best_iteration']}  {r['seconds']}s")


🏃 View run abl_tabular_only at: https://dagshub.com/abhibarnwal707/papertrail.mlflow/#/experiments/0/runs/0553eb1cb6924326bd9f86e3530beafb
🧪 View experiment at: https://dagshub.com/abhibarnwal707/papertrail.mlflow/#/experiments/0
tabular_only           nfeat=16  val_prauc=0.049  test_prauc=0.046  f1=0.116  rec@1%=0.409  p@1000=0.116  iter=84  36s
🏃 View run abl_full at: https://dagshub.com/abhibarnwal707/papertrail.mlflow/#/experiments/0/runs/47be9e0550f945f3944ebb46498d1bba
🧪 View experiment at: https://dagshub.com/abhibarnwal707/papertrail.mlflow/#/experiments/0
full                   nfeat=51  val_prauc=0.369  test_prauc=0.351  f1=0.392  rec@1%=0.629  p@1000=0.357  iter=68  137s
🏃 View run abl_full_seed1 at: https://dagshub.com/abhibarnwal707/papertrail.mlflow/#/experiments/0/runs/0545784c56fc439896332286e77d5ff0
🧪 View experiment at: https://dagshub.com/abhibarnwal707/papertrail.mlflow/#/experiments/0
full_seed1             nfeat=51  val_prauc=0.330  test_prauc=0.295  f1=0.416  rec

In [8]:
# %% Cell 6: ablation table (sorted by validation PR-AUC, because test must not be used to pick anything)
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
abl = pl.DataFrame([{
    "arm": n, "n_features": r["n_features"],
    "val_pr_auc": r["val_pr_auc"], "test_pr_auc": r["test_pr_auc"], "test_f1": r["test_f1"],
    "test_recall_1pct_fpr": r["test_recall_at_1pct_fpr"], "test_precision_1000": r["test_precision_at_1000"],
    "best_iteration": r["best_iteration"],
} for n, r in results.items()]).sort("val_pr_auc", descending=True)
print(abl)

seeds = [results[k]["test_pr_auc"] for k in ("full", "full_seed1", "full_seed2") if k in results]
if len(seeds) == 3:
    print(f"\nSEED NOISE on the full model: test PR-AUC {min(seeds):.3f} to {max(seeds):.3f} "
          f"(std {np.std(seeds):.3f}). Differences smaller than this between arms are not meaningful.")
abl.write_csv("/kaggle/working/ablation_table.csv")


shape: (15, 8)
┌────────────┬────────────┬────────────┬────────────┬──────────┬───────────┬───────────┬───────────┐
│ arm        ┆ n_features ┆ val_pr_auc ┆ test_pr_au ┆ test_f1  ┆ test_reca ┆ test_prec ┆ best_iter │
│ ---        ┆ ---        ┆ ---        ┆ c          ┆ ---      ┆ ll_1pct_f ┆ ision_100 ┆ ation     │
│ str        ┆ i64        ┆ f64        ┆ ---        ┆ f64      ┆ pr        ┆ 0         ┆ ---       │
│            ┆            ┆            ┆ f64        ┆          ┆ ---       ┆ ---       ┆ i64       │
│            ┆            ┆            ┆            ┆          ┆ f64       ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪════════════╪══════════╪═══════════╪═══════════╪═══════════╡
│ full       ┆ 51         ┆ 0.36904    ┆ 0.351189   ┆ 0.391944 ┆ 0.628661  ┆ 0.357     ┆ 68        │
│ full-graph ┆ 48         ┆ 0.349489   ┆ 0.292213   ┆ 0.392777 ┆ 0.58159   ┆ 0.317     ┆ 39        │
│ full-uniq  ┆ 47         ┆ 0.340398   ┆ 0.317775   ┆ 0.404616 ┆ 0.564854  ┆

In [10]:
# %% Cell 7: recall per typology on TEST positives (threshold = the arm's validation-chosen threshold)
patterns_path = glob.glob("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Patterns.txt", recursive=True)[0]
rows, attempt, typ = [], -1, None
with open(patterns_path) as fh:
    for line in fh:
        line = line.strip()
        if line.startswith("BEGIN"):
            attempt += 1
            typ = line.split(" - ")[1].split(":")[0].strip()
        elif line and not line.startswith("END"):
            f = line.split(",")
            rows.append((attempt, typ, f[0], f"{int(f[1])}_{f[2]}", f"{int(f[3])}_{f[4]}", float(f[7])))
patterns = (pl.DataFrame(rows, schema=["attempt_id", "typology", "ts", "src", "dst", "amt_paid"], orient="row")
              .with_columns(pl.col("ts").str.strptime(pl.Datetime, "%Y/%m/%d %H:%M")))
key = ["ts", "src", "dst", "amt_paid"]
p1 = patterns.unique(subset=key, keep="first", maintain_order=True).select(key + ["typology"])
typ_test = (test.select(key + ["label"]).join(p1, on=key, how="left")
                .with_columns(pl.when(pl.col("label") == 1).then(pl.col("typology").fill_null("UNMAPPED")).alias("typology")))
assert typ_test.height == test.height
typ_arr = typ_test["typology"].to_numpy()                  # None for negatives
pos_types = sorted({t for t in typ_arr[y_test == 1]})

show = [a for a in ("tabular_only", "full", "full-uniq", "tab+uniq", "full-graph", "tab+windowed") if a in test_preds]
trows = []
for t in pos_types:
    m = (y_test == 1) & (typ_arr == t)
    row = {"typology": t, "test_positives": int(m.sum())}
    for a in show:
        row[f"recall_{a}"] = float((test_preds[a][m] >= results[a]["threshold"]).mean())
    trows.append(row)
typ_tbl = pl.DataFrame(trows).sort("test_positives", descending=True)
print(typ_tbl)
typ_tbl.write_csv("/kaggle/working/typology_recall.csv")



shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ typology   ┆ test_posit ┆ recall_tab ┆ recall_fu ┆ recall_fu ┆ recall_ta ┆ recall_fu ┆ recall_ta │
│ ---        ┆ ives       ┆ ular_only  ┆ ll        ┆ ll-uniq   ┆ b+uniq    ┆ ll-graph  ┆ b+windowe │
│ str        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ d         │
│            ┆ i64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ ---       │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ UNMAPPED   ┆ 371        ┆ 0.013477   ┆ 0.002695  ┆ 0.002695  ┆ 0.010782  ┆ 0.002695  ┆ 0.051213  │
│ GATHER-SCA ┆ 127        ┆ 0.299213   ┆ 0.488189  ┆ 0.582677  ┆ 0.385827  ┆ 0.433071  ┆ 0.228346  │
│ TTER       ┆            ┆            ┆           ┆           ┆           ┆ 

In [11]:
# %% Cell 8: save everything for reports/
with open("/kaggle/working/ablation_metrics.json", "w") as fh:
    json.dump({"arms": results, "families": {k: v for k, v in FAMILIES.items()}, "scale_pos_weight": spw}, fh, indent=2)
print("Download ablation_table.csv, typology_recall.csv and ablation_metrics.json from the Output tab into PaperTrail/reports/")


Download ablation_table.csv, typology_recall.csv and ablation_metrics.json from the Output tab into PaperTrail/reports/
